In [1]:
class Color:
    RED = 1
    BLACK = 0

class Node:
    def __init__(self, key, color=Color.RED, left=None, right=None, parent=None):
        self.key = key
        self.color = color
        self.left = left
        self.right = right
        self.parent = parent

class RedBlackTree:
    def __init__(self):
        # Sentinel NIL node - all leaves point to this. Always black.
        self.NIL = Node(key=None, color=Color.BLACK)
        self.root = self.NIL

    def search(self, key):
        """Return node with key, or None if not found"""
        node = self.root
        while node != self.NIL:
            if key == node.key:
                return node
            node = node.left if key < node.key else node.right
        return None

    def left_rotate(self, x):
        y = x.right
        x.right = y.left
        if y.left != self.NIL:
            y.left.parent = x
        y.parent = x.parent
        if x.parent == self.NIL:
            self.root = y
        elif x == x.parent.left:
            x.parent.left = y
        else:
            x.parent.right = y
        y.left = x
        x.parent = y

    def right_rotate(self, y):
        x = y.left
        y.left = x.right
        if x.right != self.NIL:
            x.right.parent = y
        x.parent = y.parent
        if y.parent == self.NIL:
            self.root = x
        elif y == y.parent.right:
            y.parent.right = x
        else:
            y.parent.left = x
        x.right = y
        y.parent = x

    def insert(self, key):
        node = Node(key)
        node.left = node.right = self.NIL
        node.parent = self.NIL

        y = self.NIL
        x = self.root

        while x != self.NIL:
            y = x
            if node.key < x.key:
                x = x.left
            else:
                x = x.right

        node.parent = y
        if y == self.NIL:
            self.root = node
        elif node.key < y.key:
            y.left = node
        else:
            y.right = node

        node.color = Color.RED
        self._insert_fixup(node)

    def _insert_fixup(self, z):
        while z.parent.color == Color.RED:
            if z.parent == z.parent.parent.left:
                y = z.parent.parent.right  # uncle
                if y.color == Color.RED:  # Case 1
                    z.parent.color = Color.BLACK
                    y.color = Color.BLACK
                    z.parent.parent.color = Color.RED
                    z = z.parent.parent
                else:
                    if z == z.parent.right:  # Case 2
                        z = z.parent
                        self.left_rotate(z)
                    # Case 3
                    z.parent.color = Color.BLACK
                    z.parent.parent.color = Color.RED
                    self.right_rotate(z.parent.parent)
            else:
                y = z.parent.parent.left  # uncle
                if y.color == Color.RED:  # Case 1
                    z.parent.color = Color.BLACK
                    y.color = Color.BLACK
                    z.parent.parent.color = Color.RED
                    z = z.parent.parent
                else:
                    if z == z.parent.left:  # Case 2
                        z = z.parent
                        self.right_rotate(z)
                    # Case 3
                    z.parent.color = Color.BLACK
                    z.parent.parent.color = Color.RED
                    self.left_rotate(z.parent.parent)
        self.root.color = Color.BLACK

    def _transplant(self, u, v):
        if u.parent == self.NIL:
            self.root = v
        elif u == u.parent.left:
            u.parent.left = v
        else:
            u.parent.right = v
        v.parent = u.parent

    def _minimum(self, node):
        while node.left != self.NIL:
            node = node.left
        return node

    def delete(self, key):
        z = self.search(key)
        if z is None:
            return False

        y = z
        y_original_color = y.color
        if z.left == self.NIL:
            x = z.right
            self._transplant(z, z.right)
        elif z.right == self.NIL:
            x = z.left
            self._transplant(z, z.left)
        else:
            y = self._minimum(z.right)
            y_original_color = y.color
            x = y.right
            if y.parent == z:
                x.parent = y
            else:
                self._transplant(y, y.right)
                y.right = z.right
                y.right.parent = y
            self._transplant(z, y)
            y.left = z.left
            y.left.parent = y
            y.color = z.color

        if y_original_color == Color.BLACK:
            self._delete_fixup(x)
        return True

    def _delete_fixup(self, x):
        while x != self.root and x.color == Color.BLACK:
            if x == x.parent.left:
                w = x.parent.right  # sibling
                if w.color == Color.RED:  # Case 1
                    w.color = Color.BLACK
                    x.parent.color = Color.RED
                    self.left_rotate(x.parent)
                    w = x.parent.right
                if w.left.color == Color.BLACK and w.right.color == Color.BLACK:  # Case 2
                    w.color = Color.RED
                    x = x.parent
                else:
                    if w.right.color == Color.BLACK:  # Case 3
                        w.left.color = Color.BLACK
                        w.color = Color.RED
                        self.right_rotate(w)
                        w = x.parent.right
                    # Case 4
                    w.color = x.parent.color
                    x.parent.color = Color.BLACK
                    w.right.color = Color.BLACK
                    self.left_rotate(x.parent)
                    x = self.root
            else:
                w = x.parent.left  # sibling
                if w.color == Color.RED:
                    w.color = Color.BLACK
                    x.parent.color = Color.RED
                    self.right_rotate(x.parent)
                    w = x.parent.left
                if w.right.color == Color.BLACK and w.left.color == Color.BLACK:
                    w.color = Color.RED
                    x = x.parent
                else:
                    if w.left.color == Color.BLACK:
                        w.right.color = Color.BLACK
                        w.color = Color.RED
                        self.left_rotate(w)
                        w = x.parent.left
                    w.color = x.parent.color
                    x.parent.color = Color.BLACK
                    w.left.color = Color.BLACK
                    self.right_rotate(x.parent)
                    x = self.root
        x.color = Color.BLACK

    def inorder(self, node=None, result=None):
        """Helper to visualize tree contents"""
        if result is None:
            result = []
            node = self.root
        if node != self.NIL:
            self.inorder(node.left, result)
            color = "R" if node.color == Color.RED else "B"
            result.append(f"{node.key}{color}")
            self.inorder(node.right, result)
        return result


# Quick test
if __name__ == "__main__":
    rbt = RedBlackTree()

    for val in [7, 3, 18, 10, 22, 8, 11, 26, 2, 6, 13]:
        rbt.insert(val)

    print("Inorder after inserts:", rbt.inorder())

    rbt.delete(18)
    rbt.delete(11)
    rbt.delete(3)

    print("Inorder after deletes:", rbt.inorder())
    print("Search 10:", rbt.search(10) is not None)  # True
    print("Search 18:", rbt.search(18) is not None)  # False

Inorder after inserts: ['2R', '3B', '6R', '7R', '8B', '10B', '11B', '13R', '18R', '22B', '26R']
Inorder after deletes: ['2R', '6B', '7R', '8B', '10B', '13B', '22R', '26B']
Search 10: True
Search 18: False
